# Explore Player Match Data

Notebook này dùng để kiểm tra dữ liệu đã clean và join đúng chưa, trước khi train model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/processed/player_match_dataset_cleaned.csv', low_memory=False)
df['match_date'] = pd.to_datetime(df['match_date'], errors='coerce')
df['year'] = df['match_date'].dt.year

df.head()

In [ ]:
print(df.shape)
print(df.dtypes.head(20).to_string())
print(df.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
numeric_cols = [
    'rating2_all', 'acs_all', 'kills_all', 'deaths_all', 'assists_all', 'kast_all', 'adr_all', 'hsp_all', 'fb_all', 'fd_all'
]
df[numeric_cols].describe().transpose()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='map', y='rating2_all')
plt.xticks(rotation=45)
plt.title('Rating theo map')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='agent', y='acs_all')
plt.xticks(rotation=45)
plt.title('ACS theo agent')
plt.show()

In [ ]:
corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation matrix')
plt.show()

In [ ]:
# 1) Số đội tuyển và số player
n_teams = df['team_id'].nunique()
n_players = df['player_id'].nunique()
print(f'Số đội tuyển: {n_teams}')
print(f'Số player: {n_players}')

# Một số team / player nổi bật
print('\nTop 10 đội nhiều bản ghi nhất:')
print(df['team'].value_counts().head(10).to_string())
print('\nTop 10 player nhiều bản ghi nhất:')
print(df.groupby('player_id')['match_id'].nunique().sort_values(ascending=False).head(10).to_string())

In [ ]:
# 2) Có bao nhiêu player chơi trong 2-3 năm?
player_years = (
    df.dropna(subset=['match_date'])
      .groupby('player_id')['year']
      .nunique()
      .reset_index(name='years_played')
)
count_2_3_years = ((player_years['years_played'] >= 2) & (player_years['years_played'] <= 3)).sum()
print(f'Player chơi trong 2-3 năm: {count_2_3_years}')
print('\nPhân bố số năm chơi:')
print(player_years['years_played'].value_counts().sort_index().to_string())

# Biểu đồ phân bố
plt.figure(figsize=(8, 5))
sns.countplot(data=player_years, x='years_played')
plt.title('Số năm tham gia của từng player')
plt.xlabel('Số năm')
plt.ylabel('Số player')
plt.show()

In [ ]:
# 3) Thống kê cơ bản của các chỉ số player
stat_cols = [
    'rating2_all', 'acs_all', 'kills_all', 'deaths_all', 'assists_all',
    'kast_all', 'adr_all', 'hsp_all', 'fb_all', 'fd_all', 'kda_all'
]
summary = df[stat_cols].describe().T.round(2)
print(summary)

# Chỉ ra các stat có phân bố lệch nghiêng nhất
summary['std'] = df[stat_cols].std(ddof=1)
summary['mean'] = df[stat_cols].mean()
summary[['mean', 'std', 'min', '50%', 'max']].sort_values('std', ascending=False)

In [ ]:
# 4) Khám phá thêm theo map, agent, và tương quan
# Top map
print('Top 10 map:')
print(df['map'].value_counts().head(10).to_string())

# Top agent
print('\nTop 10 agent:')
print(df['agent'].value_counts().head(10).to_string())

# Rating trung bình theo map
map_rating = df.groupby('map')['rating2_all'].mean().sort_values(ascending=False)
print('\nRating trung bình theo map:')
print(map_rating.head(10).to_string())

# Heatmap tương quan giữa các stat cơ bản
corr = df[stat_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Heatmap tương quan các stat')
plt.show()

# Plot phân bố rating2_all
plt.figure(figsize=(8, 5))
sns.histplot(df['rating2_all'], bins=30, kde=True)
plt.title('Phân bố rating2_all')
plt.xlabel('rating2_all')
plt.show()